# Fleet and Discard System in PyPath

## Overview
The fleet system models fishing effort, catch allocation, and discard dynamics. This notebook demonstrates fleet creation, discard allocation, and integration with Ecosim.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypath.core.params import GearType, FleetGroup, DiscardPath, FleetDynamics

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Gear Types

Define fishing gear characteristics:

In [ ]:
# Explore gear types
print('Available Gear Types:')
for gear in GearType:
    print(f'  {gear.name}: {gear.value}')

## Part 2: Creating Fleet Groups

Define fleets and their target species:

In [ ]:
# Create a trawl fleet targeting Cod and Herring
trawl_fleet = FleetGroup(
    fleet_name='Pelagic Trawl',
    gear_type=GearType.TRAWL,
    target_species=['Cod', 'Herring', 'Sprat'],
    base_fishing_effort=1.0,
    catchability={'Cod': 0.015, 'Herring': 0.025, 'Sprat': 0.030}
)

print(f'Fleet: {trawl_fleet.fleet_name}')
print(f'Gear: {trawl_fleet.gear_type.name}')
print(f'Targets: {trawl_fleet.target_species}')
print(f'Catchability: {trawl_fleet.catchability}')

## Part 3: Discard Paths

Model where discards go (back to system or out):

In [ ]:
# Create discard path for Sprat (juveniles discarded)
sprat_discard = DiscardPath(
    species='Sprat',
    discard_rate=0.15,  # 15% of catch discarded
    destination='Detritus',  # Discards become detritus
    return_biomass=True  # Return to energy flow
)

# Create discard path for Cod (bycatch released)
cod_bycatch = DiscardPath(
    species='Cod',
    discard_rate=0.05,  # 5% bycatch
    destination='Detritus',
    return_biomass=False  # Sinks and lost
)

print(f'Sprat discard: {sprat_discard.discard_rate:.1%} -> {sprat_discard.destination}')
print(f'Cod bycatch: {cod_bycatch.discard_rate:.1%} -> {cod_bycatch.destination}')

## Part 4: Fleet Dynamics Container

Manage multiple fleets and their interactions:

In [ ]:
# Create fleet dynamics system
fleet_dynamics = FleetDynamics()

# Add fleets
fleet_dynamics.add_fleet(trawl_fleet)

# Add discard paths
fleet_dynamics.add_discard_path(sprat_discard)
fleet_dynamics.add_discard_path(cod_bycatch)

print(f'Fleet dynamics system:')
print(f'  Fleets: {fleet_dynamics.n_fleets()}')
print(f'  Discard paths: {fleet_dynamics.n_discard_paths()}')

## Part 5: Catch Allocation

Calculate catch allocation from fishing effort:

In [ ]:
# Define biomass for each species
biomass = {
    'Cod': 150.0,
    'Herring': 500.0,
    'Sprat': 200.0
}

# Calculate catch from effort
effort = 1.5  # fishing effort multiplier
catches = fleet_dynamics.allocate_catch(trawl_fleet, biomass, effort)

print('Catch allocation at 1.5x effort:')
for species, catch in catches.items():
    rate = catch / biomass[species] if species in biomass else 0
    print(f'  {species}: {catch:.2f} t ({rate:.1%} of biomass)')

## Part 6: Discard Distribution

In [ ]:
# Calculate discards from total catch
total_catch = 450.0  # t per year
discard_dist = fleet_dynamics.distribute_discard(total_catch)

# Create summary
discard_summary = pd.DataFrame([
    {'Species': 'Sprat', 'Total Catch': 140, 'Discard %': 15, 'Discard (t)': 21},
    {'Species': 'Cod', 'Total Catch': 280, 'Discard %': 5, 'Discard (t)': 14},
])

print('Discard summary:')
print(discard_summary.to_string(index=False))

## Part 7: Integration with Ecosim

Apply fleet dynamics to Ecosim fishing mortality:

In [ ]:
# Example: Fishing mortality over time
time_steps = np.array([0, 1, 2, 3, 4, 5])  # years
effort_trajectory = np.array([1.0, 1.1, 1.2, 1.3, 1.4, 1.5])

# Calculate fishing mortality trajectory
f_cod = effort_trajectory * trawl_fleet.catchability['Cod']
f_herring = effort_trajectory * trawl_fleet.catchability['Herring']

plt.figure(figsize=(10, 6))
plt.plot(time_steps, f_cod, 'o-', label='Cod F', linewidth=2)
plt.plot(time_steps, f_herring, 's-', label='Herring F', linewidth=2)
plt.xlabel('Year')
plt.ylabel('Fishing Mortality (year^-1)')
plt.title('Fishing Mortality Trajectory - Pelagic Trawl')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Part 8: DataFrame Export

In [ ]:
# Export fleet configuration
fleet_df = fleet_dynamics.to_dataframe()
print('Fleet configuration:')
print(fleet_df)

# Save to CSV
fleet_df.to_csv('fleet_config.csv', index=False)
print('\nSaved to fleet_config.csv')

## Performance Tips

- Catch allocation is O(n_species) per time step
- Cache catchability coefficients for repeated calculations
- Use vectorized operations for effort scenarios
- Pre-compute discard proportions during initialization